# core

> Fill in a module description here

Goal here is to incrementally build up to an implementation of Lisp in python. I'd like to use the solveit method starting from the inside out. So we'll likely start with literals and work our way up.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great. An inside-out path for a tiny Lisp in Python could start like this:

1. **Data/literals**: numbers, strings, booleans, symbols, lists
2. **Reader/parser**: turn text like `(+ 1 2)` into Python data structures
3. **Environment**: variable lookup and bindings
4. **Evaluator**: evaluate literals, symbols, and function calls
5. **Special forms**: `quote`, `if`, `define`, `lambda`
6. **Functions/macros later**: builtins, closures, maybe macros

For the very first step, I’d separate **self-evaluating literals** from **symbols**.

A minimal representation might be:

```python
class Symbol(str): pass
```

Then plain Python values can represent literals:

```python
42          # number literal
"hello"     # string literal
True        # boolean literal
Symbol("x") # Lisp symbol
```

The key distinction: `"x"` is a string value, but `Symbol("x")` means “look up the variable named x” once evaluation exists.

<details class='token-usage-details' markdown='1'><summary>$0.0048</summary>

`total=30,987 | in=30,751 | out=236 | cached=4.6% | $0.0048 | gpt-5.5`

</details>


help understand literals in Lisp

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In Lisp, a **literal** is a value written directly in the program.

Examples:

```lisp
42
"hello"
#t
```

These usually evaluate to themselves:

```lisp
42        ; evaluates to the number 42
"hello"   ; evaluates to the string "hello"
#t        ; evaluates to true
```

The interesting contrast is with **symbols**:

```lisp
x
```

`x` is not usually the literal string `"x"`. It means: “look up the value bound to the name `x`.”

So for our Python implementation, we can often represent Lisp literals using ordinary Python values:

```python
42, "hello", True
```

But symbols need a separate representation, because `"x"` as a Lisp string and `x` as a Lisp variable name are different things.

<details class='token-usage-details' markdown='1'><summary>$0.0048</summary>

`total=31,230 | in=31,049 | out=181 | cached=96.9% | $0.0048 | gpt-5.5`

</details>


how does Lisp distinguish between literals and symbols

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Lisp distinguishes them mostly by **syntax**.

Some things have special written forms that the reader/parser recognizes as literal values:

```lisp
42        ; number literal
"hello"   ; string literal
#t        ; boolean true, in Scheme-style Lisps
```

But a bare word like this:

```lisp
x
```

is read as a **symbol**, not a string. When evaluated, Lisp treats it as a name to look up in the environment.

So:

```lisp
"apple"   ; literal string value
apple     ; symbol/name
```

That distinction is why a Lisp-in-Python often needs something like:

```python
class Symbol(str): pass
```

so `"apple"` and `Symbol("apple")` are different kinds of things.

<details class='token-usage-details' markdown='1'><summary>$0.0048</summary>

`total=31,465 | in=31,296 | out=169 | cached=4.5% | $0.0048 | gpt-5.5`

</details>


In [ ]:
#| default_exp core

### Atoms and Literals

In [ ]:
#| export
import ast, re
from fastcore.basics import basic_repr, store_attr, first, last

In [ ]:
ast.literal_eval("123"), ast.literal_eval('"hello world"'), ast.literal_eval("2+3j")

(123, 'hello world', (2+3j))

In [ ]:
#| export
def Atom(s):
    "parse atoms, default this is a symbol"
    if s == "#t": return True
    if s == "#f": return False
    try: return ast.literal_eval(s)
    except Exception: return Symbol(s)


In [ ]:
#| export
class Symbol:
    "symbol is a `name` in Lisp"
    def __init__(self, s): store_attr()
    def __str__(self): return f"{self.s}"

    def __eq__(self, other): return isinstance(other, Symbol) and self.s == other.s
    def __hash__(self): return hash(self.s)
    
    __repr__ = basic_repr()

In [ ]:
Atom("123"), Atom('"hello world"'), Atom("2+3j"), Atom("lambda")

(123, 'hello world', (2+3j), Symbol(s='lambda'))

In [ ]:
Atom("+") == Atom("+")

True

In [ ]:
Atom("+") == Atom('"+"')

False

like that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — that’s exactly the right shape.

You’ve made the key distinction:

```python
Atom("+")     # Symbol(s='+')
Atom('"+"')   # "+"
```

So bare `+` means “the symbol named `+`”, while quoted `"+"` means “the string containing plus”. Nice.

One small design note: lots of Lisps make `Symbol` a subclass of `str`, but your wrapper class is also totally fine, especially while learning. Its explicit `s` field makes the distinction very clear.

<details class='token-usage-details' markdown='1'><summary>$0.0049</summary>

`total=32,274 | in=32,161 | out=113 | cached=20.3% | $0.0049 | gpt-5.5`

</details>


### Reader: Lexer and Parser
parse strings into lisp expressions

In [ ]:
#| export
STR = r'"(?:\\.|[^"\\])*"'  # string with spaces etc.
COMMENT = r';[^\n]*'
ATOM = r'''[^\s()`',;]+'''

UNQ_SPL = ',@'              # unquote splice
PAREN = '[()]'
QUOTE = "[`',]"

# order matters `,@` needs to show up before the single character `,`
TOKEN_RE = "|".join([STR, COMMENT, UNQ_SPL, PAREN, QUOTE, ATOM])

def lexer(s): return [t for t in re.findall(TOKEN_RE, s) if not t.startswith(';')]

In [ ]:
lexer("( + 1 2 )")

['(', '+', '1', '2', ')']

In [ ]:
lexer("'( + 1 2 )")

["'", '(', '+', '1', '2', ')']

In [ ]:
lexer("`( + 1 2 )")

['`', '(', '+', '1', '2', ')']

In [ ]:
lexer('`(+ ,@xs 2)')

['`', '(', '+', ',@', 'xs', '2', ')']

In [ ]:
#| export
def parser(toks):
    if not toks: raise SyntaxError("malformed list: unexpected EOF")
    

    t = toks.pop(0)
    if t == ')': raise SyntaxError("malformed list: unexpected )")
    if t == '(':
        sl = []
        while toks and toks[0] != ')': sl.append(parser(toks))
        if not toks: raise SyntaxError("malformed list: missing )")
        toks.pop(0)
        return sl
    sugar = {
        "'": "quote",
        "`": "quasiquote",
        ",": "unquote",
        ",@": "unquote-splicing",
    }
    if t in sugar: return [Symbol(sugar[t]), parser(toks)]

    return Atom(t)

In [ ]:
parser(lexer('`(+ ,@xs 2)'))

[Symbol(s='quasiquote'),
 [Symbol(s='+'), [Symbol(s='unquote-splicing'), Symbol(s='xs')], 2]]

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yep, this is a good recursive parser.

It now has the classic Lisp reader structure:

- `(` starts a list
- `)` is only valid when closing an existing list
- quote syntax gets expanded into ordinary list forms:

```python
'(+ 1 2)
# becomes roughly:
[Symbol("quote"), [Symbol("+"), 1, 2]]
```

One small improvement I’d consider soon: wrap this in a `read(s)` function so callers don’t mutate token lists directly:

```python
def read(s): return parser(lexer(s))
```

Then you can test with `read('(+ 1 2)')`.

<details class='token-usage-details' markdown='1'><summary>$0.0051</summary>

`total=33,678 | in=33,539 | out=139 | cached=95.8% | $0.0051 | gpt-5.5`

</details>


In [ ]:
#| export
def parse(s): 
    toks = lexer(s)
    r = parser(toks)
    if not toks: return r
    raise SyntaxError("unexpected tokens after expression")
    

In [ ]:
parse('`(+ ,@xs 2)')

[Symbol(s='quasiquote'),
 [Symbol(s='+'), [Symbol(s='unquote-splicing'), Symbol(s='xs')], 2]]

In [ ]:
parse('`(+ ,@xs 2)')

[Symbol(s='quasiquote'),
 [Symbol(s='+'), [Symbol(s='unquote-splicing'), Symbol(s='xs')], 2]]

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice — `parse` is the right wrapper.

It does two important things:

1. hides the mutable `toks.pop(0)` detail
2. rejects extra trailing forms, e.g. something like:

```python
parse("(+ 1 2) 99")
```

should raise `"unexpected tokens after expression"`.

At this point you’ve got enough reader/parser machinery to start the first tiny `eval`: literals self-evaluate, symbols look up in an environment, lists mean “call something.”

<details class='token-usage-details' markdown='1'><summary>$0.0052</summary>

`total=33,936 | in=33,828 | out=108 | cached=98.0% | $0.0052 | gpt-5.5`

</details>


In [ ]:
#| export
def parse_all(s):
    toks = lexer(s)
    exprs = []
    while toks: exprs.append(parser(toks))
    return exprs

In [ ]:
parse_all("""
(define x 10)
(+ x 5)
""")

[[Symbol(s='define'), Symbol(s='x'), 10], [Symbol(s='+'), Symbol(s='x'), 5]]

### The Evaluator: eval and apply

help me understand how eval and apply work together?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Think of `eval` and `apply` as two partners:

- **`eval` answers:** “What does this expression mean?”
- **`apply` answers:** “Given a function and some already-evaluated arguments, what result do we get?”

For a Lisp expression like:

```scheme
(+ 1 (* 2 3))
```

`eval` starts with the whole list:

```scheme
(+ 1 (* 2 3))
```

It sees a list, so it:

1. evaluates `+` to find the function
2. evaluates `1` to get `1`
3. evaluates `(* 2 3)` to get `6`
4. asks `apply` to call `+` on `[1, 6]`

So the rough split is:

```python
eval(expr, env)   # turns expressions into values
apply(fn, args)   # calls functions on values
```

The important idea: **`eval` recursively prepares things; `apply` performs the actual call.**

For normal function calls, `eval` evaluates all the arguments first. Later, special forms like `if`, `define`, and `quote` will be exceptions, because they do *not* evaluate everything in the usual way.

<details class='token-usage-details' markdown='1'><summary>$0.0054</summary>

`total=34,878 | in=34,542 | out=336 | cached=18.9% | reasoning=69 | $0.0054 | gpt-5.5`

</details>


In [ ]:
#| export
def scm_apply(fn, args): return fn(*args)

like that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — for this first stage, that’s perfect:

```python
def scm_apply(fn, args): return fn(*args)
```

It says: once `eval` has already found a Python callable and evaluated the arguments, `apply` just calls it.

Later, `scm_apply` will probably grow to handle user-defined Scheme functions too, but this is exactly the right minimal version.

<details class='token-usage-details' markdown='1'><summary>$0.0053</summary>

`total=35,003 | in=34,919 | out=84 | cached=97.9% | $0.0053 | gpt-5.5`

</details>


In [ ]:
def scm_eval(expr, env, sfs=()):
    "eval-uate lisp expressions given an environment and handlers for special forms"
    # Atoms
    if isinstance(expr, Symbol): return env[expr.s]
    if not isinstance(expr, list): return expr

    # lists
    if expr == []: return []
    hd, *body = expr

    if isinstance(hd, Symbol) and hd.s in sfs: return sfs[hd.s](body, env, sfs)

    return scm_apply(scm_eval(hd, env, sfs), [scm_eval(o, env, sfs) for o in body])


like that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — that’s a strong minimal `eval`.

It has the right core cases:

- `Symbol` → look up in `env`
- non-list literal → self-evaluate
- empty list → returns itself
- special form → delegate before evaluating arguments
- normal list → evaluate head and args, then `scm_apply`

One tiny naming thought: `sfs` is fine, but something like `special_forms` may be easier to read later. The logic itself looks good.

<details class='token-usage-details' markdown='1'><summary>$0.0052</summary>

`total=34,563 | in=34,459 | out=104 | cached=18.9% | $0.0052 | gpt-5.5`

</details>


### Primitive Procedures
Give the interpreter some basic skills.

In [ ]:
#| export
import math, operator as op

def scm_sub(x, *xs): return x - sum(xs) if xs else -x
def scm_div(x, *xs): return 1/x if not xs else x / math.prod(xs)

def is_num(x): return not isinstance(x, bool) and isinstance(x, (int, float, complex))
def is_sym(x): return isinstance(x, Symbol)

_builtin = {
    "+": lambda *xs: sum(xs),
    "*": lambda *xs: math.prod(xs), 
    "/": scm_div,
    "-": scm_sub,
    ">": op.gt, 
    "<": op.lt, 
    "=": lambda x,y: is_num(x) and is_num(y) and x == y,
    "<=": op.le,
    ">=": op.ge,
    
    "number?": is_num,
    "string?": lambda x: isinstance(x, str),
    "symbol?": is_sym,
    "boolean?": lambda x: isinstance(x, bool),
    "null?": lambda x: x == [],
    "pair?": lambda x: isinstance(x, list) and len(x) > 0,
    "list?": lambda x: isinstance(x, list),

}


In [ ]:
scm_eval(parse("(+ 1 2)"), dict(_builtin))

3

In [ ]:
scm_eval(parse("(+ 1 (* 2 3 4))"), dict(_builtin))

25

In [ ]:
scm_eval(parse("(- 5)"), dict(_builtin))

-5

In [ ]:
scm_eval(parse("(+ 1 (- 5))"), dict(_builtin))

-4

In [ ]:
scm_eval(parse("(- 10 3 2)"), dict(_builtin))

5

In [ ]:
scm_eval(parse("(/ 2 5 6)"), dict(_builtin))

0.06666666666666667

### Special Forms

The big difference is that arguments for special forms are passed in directly without evaluation. It is up to the the special form to interpret them.

For instance:
```
(if #t 1 (/ 1 0))
```
The `else` clause should not be greedily evaluated. `if` needs to figure out if it wants to handle it.

#### quote
```lisp
    `(+ 1 2) -> (+ 1 2)
```

In [ ]:
#| export
def _sf_quote(xs, env, sfs): return xs[0]

In [ ]:
scm_eval(parse("'(+ 1 2)"), dict(_builtin), sfs={"quote": _sf_quote})

[Symbol(s='+'), 1, 2]

note this returns the list rather than the evaluated version of that list (1+2 = 3)

#### define — binding symbols
```lisp
    (define name value)

    (define square (lambda (x) (* x x)))
```

does define always take two arguments?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In the simplest core Lisp/Scheme, yes:

```scheme
(define name value)
```

so it has two parts after `define`: the symbol to bind, and the expression whose value gets stored.

But Scheme also has a shorthand for defining functions:

```scheme
(define (square x) (* x x))
```

which is really shorthand for something like:

```scheme
(define square (lambda (x) (* x x)))
```

For your current interpreter, I’d start with only the simple two-argument form.

<details class='token-usage-details' markdown='1'><summary>$0.0056</summary>

`total=36,873 | in=36,760 | out=113 | cached=98.5% | $0.0056 | gpt-5.5`

</details>


In [ ]:
def _sf_define(xs, env, sfs):
    sym = xs[0]
    if not isinstance(sym, Symbol): raise SyntaxError(f"Define argument {sym} must be a symbol")
    
    env[sym.s] = scm_eval(xs[1], env, sfs)
    return sym

In [ ]:
global_env = dict(_builtin)
scm_eval(parse("(define a 41)"), global_env, sfs={"define": _sf_define})

global_env

{'+': <function __main__.<lambda>(*xs)>,
 '*': <function __main__.<lambda>(*xs)>,
 '/': <function __main__.scm_div(x, *xs)>,
 '-': <function __main__.scm_sub(x, *xs)>,
 '>': <function _operator.gt(a, b, /)>,
 '<': <function _operator.lt(a, b, /)>,
 '=': <function __main__.<lambda>(x, y)>,
 '<=': <function _operator.le(a, b, /)>,
 '>=': <function _operator.ge(a, b, /)>,
 'number?': <function __main__.is_num(x)>,
 'string?': <function __main__.<lambda>(x)>,
 'symbol?': <function __main__.is_sym(x)>,
 'boolean?': <function __main__.<lambda>(x)>,
 'null?': <function __main__.<lambda>(x)>,
 'pair?': <function __main__.<lambda>(x)>,
 'list?': <function __main__.<lambda>(x)>,
 'a': 41}

In [ ]:
scm_eval(parse("a"), global_env)

41

should we create an environment next?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — that’s a great next step.

Right now `env` is just a plain dict, which is enough for global names like:

```scheme
(define a 41)
```

But soon, for `lambda`, you’ll need **nested environments**:

- look in the local function environment first
- if not found, look in the parent/global environment

So the next useful abstraction is an `Env` that behaves like a dict but has an optional parent. Conceptually:

```python
class Env(dict):
    def find(self, k): return self if k in self else self.parent.find(k)
```

Then symbol lookup becomes “find the environment containing this name, then get it.”

<details class='token-usage-details' markdown='1'><summary>$0.0057</summary>

`total=36,811 | in=36,505 | out=306 | cached=93.6% | reasoning=157 | $0.0057 | gpt-5.5`

</details>


In [ ]:
#| export
class Env(dict):
    def __init__(self, bindings=(), parent=None, **kw):
        super().__init__(**kw)
        self.update(dict(bindings))
        self.parent = parent

    def __missing__(self, k): 
        if self.parent is not None: return self.parent[k]
        raise KeyError(k)

    def find(self, s):
        if s in self: return self
        if self.parent is not None: return self.parent.find(s)
        raise KeyError(s)
   
    def new_frame(self, bindings=()):
        return Env(bindings, parent=self)

like that?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — this is a good `Env` for lexical scoping.

The key pieces are right:

- `__missing__` lets normal lookup fall through to the parent
- `find` returns the actual frame where a name lives, useful for later `set!`
- `new_frame` creates a child environment for function calls

One small thing to watch: `dict(bindings)` means bindings need to be pairs like:

```python
[("x", 3), ("y", 4)]
```

which is exactly what you’ll want when binding lambda parameters to argument values.

<details class='token-usage-details' markdown='1'><summary>$0.0055</summary>

`total=36,213 | in=36,089 | out=124 | cached=3.9% | $0.0055 | gpt-5.5`

</details>


so is the next thing lambda?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Yes — `lambda` is the natural next big step.

You already have the pieces it needs:

- `Env` can support local scopes
- `define` can bind names
- `scm_apply` can be extended beyond Python callables

Conceptually, evaluating:

```scheme
(lambda (x) (+ x 1))
```

should produce a **function object** that remembers:

1. parameter names: `x`
2. body: `(+ x 1)`
3. the environment where it was created

That last part is important: it’s what makes closures work.

So next you probably want a small `Procedure`/`Func` object, then update `scm_apply` to handle either Python functions or your Lisp functions.

<details class='token-usage-details' markdown='1'><summary>$0.0058</summary>

`total=37,441 | in=37,145 | out=296 | cached=92.0% | reasoning=137 | $0.0058 | gpt-5.5`

</details>


#### lambda — anonymous procedures

```lisp
    (lambda (x) (+ 2 x)
```

In [ ]:
def _sf_lambda(xs, env, sfs):
    params, body = xs
    def procedure(*args):
        return scm_eval(body, env.new_frame(zip([o.s for o in params], args)), sfs)
    return procedure

In [ ]:
scm_eval(parse("((lambda (x) (+ 2 x)) 40)"), Env(**_builtin), sfs={"lambda": _sf_lambda})


42

#### if

In [ ]:
def _sf_if(xs, env, sfs):
    cond,then_,else_ = xs
    return scm_eval(else_, env, sfs) if scm_eval(cond, env, sfs) is False else scm_eval(then_, env, sfs)

In [ ]:

scm_eval(parse("(if #t 1 (/ 1 0))"), Env(**_builtin), sfs={"if": _sf_if})

1

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Nice — this is exactly why `if` must be a special form.

Your test proves the important behavior:

```scheme
(if #t 1 (/ 1 0))
```

returns `1`, because the `else` branch is never evaluated. If `if` were an ordinary builtin, `(/ 1 0)` would be evaluated before `if` ever got called.

<details class='token-usage-details' markdown='1'><summary>$0.0058</summary>

`total=38,050 | in=37,967 | out=83 | cached=98.1% | $0.0058 | gpt-5.5`

</details>


#### begin

```lisp
    (begin
        (define a 2)
        (+ a 2))
```

In [ ]:
def _sf_begin(xs, env, sfs):
    r = None
    for o in xs: r = scm_eval(o, env, sfs)
    return r

In [ ]:
scm_eval(parse("""
(begin
  (define a 2)
  (+ a 2))
"""), Env(**_builtin), sfs={"begin":_sf_begin, "define":_sf_define})

4

In [ ]:
scm_eval(parse("""
(begin
  (define add2 (lambda (x) (+ x 2)))
  (add2 40))
"""), Env(**_builtin), sfs={"quote":_sf_quote, "define":_sf_define, "lambda":_sf_lambda, "if":_sf_if, "begin":_sf_begin})

42

In [ ]:
scm_eval(parse("""

(begin
  (define fact (lambda (n)
    (if (= n 0) 1 (* n (fact (- n 1))))))
  (fact 5))

"""), Env(**_builtin), sfs={"quote":_sf_quote, "define":_sf_define, "lambda":_sf_lambda, "if":_sf_if, "begin":_sf_begin})

120

.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Great — recursion works. That’s a big milestone.

And yes, now TCO becomes relevant: your factorial works for small `n`, but deep recursion will eventually hit Python’s recursion limit. The next conceptual step is to make `scm_eval` iterative for tail positions, usually by returning or looping over “the next expression/environment to evaluate” instead of recursively calling Python in tail position.

<details class='token-usage-details' markdown='1'><summary>$0.0060</summary>

`total=39,416 | in=39,333 | out=83 | cached=97.3% | $0.0060 | gpt-5.5`

</details>


### Tail-Call Optimization
The factorial method above will stack overflow for deep recursion. Instead of recursing, we can evaluate iteratively

In [ ]:
#| export
class Thunk:
    "a piece of computation that can be executed later"
    def __init__(self, expr, env): store_attr()

scm_eval_one_step and scm_eval_tco recursively call each other.  


One step returns expression, if it is a thunk, it gets executed in the scm_eval_tco. This avoids having deep stacks.

In [ ]:
#| export
class Macro:
    def __init__(self, fn): store_attr()

In [ ]:
#| export
def scm_eval_one_step(expr, env, sfs=()):
    "eval-uate lisp expressions given an environment and handlers for special forms"
    # Atoms
    if isinstance(expr, Symbol): return env[expr.s]
    if not isinstance(expr, list): return expr

    # lists
    if expr == []: return []
    hd, *body = expr

    if isinstance(hd, Symbol) and hd.s in sfs: return sfs[hd.s](body, env, sfs)
    fn = scm_eval_tco(hd, env, sfs)
    if isinstance(fn, Macro): return Thunk(fn.fn(*body), env)

    return scm_apply(fn, [scm_eval_tco(o, env, sfs) for o in body])

In [ ]:
#| export
def scm_eval_tco(expr, env, sfs=()):
    "eval but with tail call optimization"
    while True:
        r = scm_eval_one_step(expr, env, sfs)
        if isinstance(r, Thunk): expr, env = r.expr, r.env
        else: return r

Now, we need TCO versions of special forms that return Thunks which could be iteratively evaluated rather than recurse into it

#### begin (TCO)
```lisp
    (begin 
        (define x 5) 
        (+ x 2))
```

In [ ]:
#| export
def _sf_begin_tco(xs, env, sfs):
    for o in xs[:-1]: scm_eval_tco(o, env, sfs)
    return Thunk(xs[-1], env)

#### if (TCO)

```lisp
    (if #t 1 (/ 1 0)
```

In [ ]:
#| export
def _sf_if_tco(xs, env, sfs):
    cond,then_,else_ = xs
    br = else_ if scm_eval_tco(cond, env, sfs) is False else then_
    return Thunk(br, env)


In [ ]:
#| export
def body_expr(body): return body[0] if len(body) == 1 else [Symbol("begin"), *body]

In [ ]:
#| export
def arity(ps, vs):
    if len(vs) == len(ps): return vs
    raise ValueError(f"incorrect number of vs passed (wanted: {len(ps)}, got: {len(vs)})")

In [ ]:
#| export
def bind_params(ps, vs):
    if isinstance(ps, Symbol): return {ps.s: list(vs)}
    return dict(zip([p.s for p in ps], arity(ps, vs)))

#### lambda (TCO)

```lisp

    (lambda (x) (+ x 2))

```

In [ ]:
#| export
def _sf_lambda_tco(xs, env, sfs): 
    params, *body = xs
    def procedure(*args):
        return Thunk(body_expr(body), env.new_frame(bind_params(params, args)))
    return procedure
    # return lambda *args: Thunk(body_expr(body), Env(**bind_params(params, args), parent=env))

#### define (TCO)

```lisp
    (define name value)

    (define square (lambda (x) (* x x)))

    (define (square x) (* x x))
```

In [ ]:
#| export
def _sf_define_tco(xs, env, sfs):
    def _mkfn(sym, expr): 
        if not isinstance(sym, Symbol): raise SyntaxError(f"{sym} must be a symbol")
        env[sym.s] = scm_eval_tco(expr, env, sfs)
        return sym.s

    arg0, *rest = xs
    if isinstance(arg0, Symbol): return _mkfn(arg0, body_expr(rest))
    if isinstance(arg0, list): return _mkfn(arg0[0], [Symbol("lambda"), arg0[1:], *rest])


we also added support for short form of define here

In [ ]:
scm_eval_tco(
    parse("""
        (begin
          (define fact-iter
            (lambda (n acc)
            (if (= n 0) acc (fact-iter (- n 1) (* acc n)))))
          (fact-iter 5 1))
    """), 
    Env(**_builtin), 
    sfs={"quote":_sf_quote, "define":_sf_define_tco, "lambda":_sf_lambda_tco, "if":_sf_if_tco, "begin":_sf_begin_tco}
)

120

In [ ]:
scm_eval_tco(
    parse("""

(begin
  (define fact-iter
    (lambda (n acc)
    (if (= n 0) acc (fact-iter (- n 1) (* acc n)))))
  (fact-iter 1000 1))        
  
    """), 
    Env(**_builtin), 
    sfs={"quote":_sf_quote, "define":_sf_define_tco, "lambda":_sf_lambda_tco, "if":_sf_if_tco, "begin":_sf_begin_tco}
)

4023872600770937735437024339230039857193748642107146325437999104299385123986290205920442084869694048004799886101971960586316668729948085589013238296699445909974245040870737599188236277271887325197795059509952761208749754624970436014182780946464962910563938874378864873371191810458257836478499770124766328898359557354325131853239584630755574091142624174743493475534286465766116677973966688202912073791438537195882498081268678383745597317461360853795345242215865932019280908782973084313928444032812315586110369768013573042161687476096758713483120254785893207671691324484262361314125087802080002616831510273418279777047846358681701643650241536913982812648102130927612448963599287051149649754199093422215668325720808213331861168115536158365469840467089756029009505376164758477284218896796462449451607653534081989013854424879849599533191017233555566021394503997362807501378376153071277619268490343526252000158885351473316117021039681759215109077880193931781141945452572238655414610628921879602238389714760

### Primitive List Operations

In [ ]:

list_primitives = {
    "cons": lambda x,y: [x] + y,
    "car": lambda xs: first(xs),
    "cdr": lambda xs: xs[1:],
    "list": lambda *xs: list(xs),
}

In [ ]:
scm_eval_tco(parse("(car (cdr '(10 20 30)))"), Env({**_builtin, **list_primitives}), sfs={"quote":_sf_quote})

20

In [ ]:
_builtin | list_primitives

{'+': <function __main__.<lambda>(*xs)>,
 '*': <function __main__.<lambda>(*xs)>,
 '/': <function __main__.scm_div(x, *xs)>,
 '-': <function __main__.scm_sub(x, *xs)>,
 '>': <function _operator.gt(a, b, /)>,
 '<': <function _operator.lt(a, b, /)>,
 '=': <function __main__.<lambda>(x, y)>,
 '<=': <function _operator.le(a, b, /)>,
 '>=': <function _operator.ge(a, b, /)>,
 'number?': <function __main__.is_num(x)>,
 'string?': <function __main__.<lambda>(x)>,
 'symbol?': <function __main__.is_sym(x)>,
 'boolean?': <function __main__.<lambda>(x)>,
 'null?': <function __main__.<lambda>(x)>,
 'pair?': <function __main__.<lambda>(x)>,
 'list?': <function __main__.<lambda>(x)>,
 'cons': <function __main__.<lambda>(x, y)>,
 'car': <function __main__.<lambda>(xs)>,
 'cdr': <function __main__.<lambda>(xs)>,
 'list': <function __main__.<lambda>(*xs)>}

### let — lexical bindings
```
    (let ((x 40))
      (+ x 2))

```

In [ ]:
def _sf_let(xs, env, sfs):
    binds, *body = xs
    new_env = Env(parent=env, **{name.s: scm_eval_tco(val, env, sfs) for name, val in binds})
    return Thunk(body_expr(body), new_env)

### Macros
```
    (define unless 
      (macro (test body) `(if ,test #f ,body)))

```

In [ ]:
#| export
class Macro:
    def __init__(self, fn): store_attr()

In [ ]:
#| export
def _macro(args, env, sfs):
    params, *body = args
    return Macro(lambda *vals: scm_eval_tco(body_expr(body), Env(parent=env, **bind_params(params, vals)), sfs))

In [ ]:
#| export
class Matcher(Symbol):
    def match(self, x): return isinstance(x, list) and len(x) == 2 and x[0] == self
    def val(self, x): return x[1] if self.match(x) else None

QQ, UQ, UQS = list(map(Matcher, ["quasiquote", "unquote", "unquote-splicing"]))

def qq(x, env, sfs, depth=1):
    "handle ` , and ,@"
    if not isinstance(x, list): return x
    if QQ.match(x): return [QQ, qq(x[1], env, sfs, depth+1)]
    if UQ.match(x): return scm_eval_tco(x[1], env, sfs) if depth == 1 else [UQ, qq(x[1], env, sfs, depth-1)]
    
    if UQS.match(x):
        if depth == 1: raise SyntaxError("unquote-splicing only valid inside a list")
        return [UQS, qq(x[1], env, sfs, depth-1)]
    

    res = []
    for o in x:
        if UQS.match(o) and depth == 1: res.extend(scm_eval_tco(o[1], env, sfs))
        else: res.append(qq(o, env, sfs, depth))
    
    return res

In [ ]:
#| export
def _sf_quasiquote(args, env, sfs): return qq(args[0], env, sfs)

In [ ]:
scm_eval_tco(parse("(let ((x 42) (xs '(1 2 3))) `(a ,x ,@xs z))"),
             Env({**_builtin, **list_primitives}),
             sfs={"quote":_sf_quote, "quasiquote":_sf_quasiquote, "let":_sf_let})

[Symbol(s='a'), 42, 1, 2, 3, Symbol(s='z')]

In [ ]:
scm_eval_tco(parse("(begin (define unless (macro (test body) `(if ,test #f ,body))) (unless (= 1 2) 42))"),
             Env({**_builtin, **list_primitives}),
             sfs={"quote":_sf_quote, "quasiquote":_sf_quasiquote, "macro":_macro, "define":_sf_define_tco, "if":_sf_if_tco, "begin":_sf_begin_tco})

42

In [ ]:
def _sf_set(xs, env, sfs):
    sym, expr = xs
    if not isinstance(sym, Symbol): raise SyntaxError(f"set! argument {sym} must be a symbol")
    env.find(sym.s)[sym.s] = scm_eval_tco(expr, env, sfs)
    return sym.s

In [ ]:
scm_eval_tco(parse("""
(begin
  (define x 1)
  (set! x 42)
  x)
"""), Env(**_builtin, **list_primitives), sfs={"begin": _sf_begin_tco, "define": _sf_define_tco, "set!": _sf_set})

42

what are the gaps from this version and R5RS ?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Quite a lot — but in a good way. You’ve built a solid **core evaluator**, not a full R5RS Scheme yet.

Big gaps from R5RS:

- **Data model**
  - no proper cons cells / dotted pairs yet
  - lists are Python lists, so `(cons 1 2)` cannot produce `(1 . 2)`
  - no characters, vectors, ports, exact/inexact number tower, rational numbers, etc.

- **Reader syntax**
  - no dotted list syntax: `'(a . b)`
  - no vectors: `#(1 2 3)`
  - no characters: `#\a`, `#\space`
  - string parsing is Python-ish via `ast.literal_eval`, not fully Scheme syntax

- **Special forms**
  Missing many R5RS forms, such as:

```scheme
cond case and or
let* letrec named let
do delay
```

- **Procedures / standard library**
  You only have a small primitive set. R5RS includes many more:
  - numeric predicates and operations
  - pair/list operations: `append`, `map`, `for-each`, `reverse`, etc.
  - equality: `eq?`, `eqv?`, `equal?`
  - string/vector/symbol conversion functions
  - I/O procedures

- **Macros**
  Your `macro` is more like a classic unhygienic Lisp macro.
  R5RS standard macros are based on hygienic `syntax-rules`.

- **Continuations**
  R5RS has:

```scheme
call-with-current-continuation
call/cc
```

  That’s a big feature and usually nontrivial in Python.

- **Tail calls**
  R5RS requires proper tail recursion everywhere. Your thunk-based TCO is a good start, but you’d still want to audit all tail positions carefully.

- **Error behavior / compliance**
  R5RS specifies lots of subtle behavior around arity, mutation, equivalence, environments, and unspecified return values. Your interpreter currently uses Python exceptions and Python values directly, which is perfect for learning but not spec-complete.

The biggest conceptual gap is probably this: **R5RS Scheme has pairs as the fundamental compound data type; your interpreter currently uses Python lists.** That’s fine for now, but implementing real cons cells would make it feel much more Scheme-like.

<details class='token-usage-details' markdown='1'><summary>$0.0072</summary>

`total=43,816 | in=42,795 | out=1,021 | cached=15.3% | reasoning=516 | $0.0072 | gpt-5.5`

</details>


ok so how would we think about the fundamental compound data type 

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

In Scheme, the fundamental compound value is not “list” — it is the **pair**.

A pair is just two slots:

```python
Pair(car, cdr)
```

Scheme calls those slots:

- `car`: the first part
- `cdr`: the rest part

A proper list is a chain of pairs ending in the empty list:

```scheme
'(1 2 3)
```

is structurally like:

```scheme
(cons 1 (cons 2 (cons 3 '())))
```

So conceptually:

```text
Pair(1, Pair(2, Pair(3, [])))
```

The important bit: the `cdr` does **not** have to be another list. That means Scheme can represent dotted pairs:

```scheme
(cons 1 2)
```

as:

```text
Pair(1, 2)
```

which prints as:

```scheme
(1 . 2)
```

So the shift is:

> A list is a special kind of pair-chain, not the primitive structure itself.

Right now your interpreter uses Python `list` for both Lisp source expressions and Lisp runtime lists. That’s simple and useful, but to become more Scheme-like, you’d separate:

- **reader/evaluator syntax lists**: maybe still Python lists internally
- **Scheme runtime pairs/lists**: represented with your own `Pair` class plus `[]`/`nil` for empty list

That unlocks real `cons`, `car`, `cdr`, dotted lists, and Scheme-style list identity/mutation later.

<details class='token-usage-details' markdown='1'><summary>$0.0069</summary>

`total=43,846 | in=43,367 | out=479 | cached=15.1% | reasoning=149 | $0.0069 | gpt-5.5`

</details>


why is this helpful?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

It’s helpful because **pairs are more general than lists**.

With Python lists, every compound thing is forced to be a proper sequence:

```python
[1, 2, 3]
```

But Scheme’s `cons` can build:

```scheme
(cons 1 2)      ; a pair, not a list
(cons 1 '())    ; a proper one-element list
```

So pairs let you represent both:

- ordinary lists: `(1 2 3)`
- dotted pairs: `(1 . 2)`

They also make Scheme operations more fundamental:

```scheme
(car (cons 1 2)) ; 1
(cdr (cons 1 2)) ; 2
```

So instead of treating `cons`, `car`, and `cdr` as list tricks, they become the primitive building blocks for compound data.

<details class='token-usage-details' markdown='1'><summary>$0.0067</summary>

`total=44,043 | in=43,860 | out=183 | cached=98.9% | $0.0067 | gpt-5.5`

</details>


In [ ]:
#| export
class Pair:
    "a cons cell"
    __slots__ = ("car", "cdr")
    def __init__(self, car, cdr): self.car,self.cdr = car,cdr

    def __eq__(self, other): return isinstance(other, Pair) and self.car == other.car and self.cdr == other.cdr

    __repr__ = basic_repr("car,cdr")

class NilType:
    def __repr__(self): return "()"

Nil = NilType()

In [ ]:
xs = Pair(1, Pair(2, Pair(3, Nil)))

In [ ]:
xs

Pair(car=1, cdr=Pair(car=2, cdr=Pair(car=3, cdr=())))

In [ ]:
#| export
cons = lambda x,y: Pair(x, y)
car = lambda x: x.car
cdr = lambda x: x.cdr

In [ ]:
car(xs)

1

In [ ]:
car(cdr(xs))

2

In [ ]:
car(cdr(cdr(xs)))

3

In [ ]:
cdr(cdr(cdr(xs)))

()

In [ ]:
cons(0, xs)

Pair(car=0, cdr=Pair(car=1, cdr=Pair(car=2, cdr=Pair(car=3, cdr=()))))

In [ ]:
cons(1, 2)

Pair(car=1, cdr=2)

In [ ]:
def list2pair(lst): 
    if not isinstance(lst, list): return lst
    return Nil if not lst else Pair(list2pair(lst[0]), list2pair(lst[1:]))

In [ ]:
list2pair([1, 2, 3, 4])

Pair(car=1, cdr=Pair(car=2, cdr=Pair(car=3, cdr=Pair(car=4, cdr=()))))

In [ ]:
list2pair([1, 2, [99, 100], 4])

Pair(car=1, cdr=Pair(car=2, cdr=Pair(car=Pair(car=99, cdr=Pair(car=100, cdr=())), cdr=Pair(car=4, cdr=()))))

In [ ]:
def _sf_quote(xs, env, sfs): return list2pair(xs[0])

### nbdev postscript

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()